# Trajectory Compliance Evaluation on Free-Tier TPU (Part 2 Demo)

This notebook is the free-tier companion to Part 2 of the tutorial series. It runs the
**trajectory evaluation pattern** from `05_trajectory_eval/` end to end on a small model
(`google/gemma-3-1b-it`) over **20 staged demo trajectories**, on the TPU runtime Colab
gives away.

**This is a demo, not the experiment.** The pattern here is identical to the measured
setup (canonical records, per-state judge prompts, structured outputs, a rules baseline
to compare against), and the model and hardware are deliberately smaller. **No benchmark
claims are made in this notebook**, and none of its outputs feed the published tables.
The measured numbers come from the GCE `v5e-4` path described in the repo README and
`05_trajectory_eval/README.md`.

> **Heads up on Colab quotas**: free-tier Colab gates TPU access pretty aggressively. If
> you see "Cannot connect to TPU backend due to usage limits," you've exhausted your
> daily allocation. Wait 24 hours for the rolling reset, switch Google accounts, or
> consider [Kaggle Notebooks](https://www.kaggle.com/code) which offer 30 hours/week of
> TPU v3-8 free. The code itself runs on any TPU generation; only the provisioning
> changes.

Runtime: **Runtime -> Change runtime type -> TPU** before running anything.


In [ ]:
# Confirm a TPU runtime is attached before installing anything.
import os

assert (
    "COLAB_TPU_ADDR" in os.environ or os.environ.get("TPU_NAME") or os.path.exists("/dev/accel0")
), "No TPU detected. Runtime -> Change runtime type -> TPU, then restart and rerun."
print("TPU runtime detected.")

In [ ]:
# Install the TPU-specific vLLM package (pip path, no Docker on Colab).
# vllm-tpu is a separate PyPI package from the CUDA 'vllm'; installing the
# wrong one is the most common failure on this path.
%pip install --quiet vllm-tpu
print("vllm-tpu installed. If Colab suggests a runtime restart, do it, then rerun from here.")

In [ ]:
# google/gemma-3-1b-it is a gated checkpoint: accept the license on its
# Hugging Face page first, then authenticate here. The token stays in this
# Colab session.
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Fetch the Part 2 module and the staged demo trajectories from the repo.
import sys

!git clone --quiet --depth 1 https://github.com/ByteanAtomResearch/compliance-at-scale-tpu.git
%cd compliance-at-scale-tpu

sys.path.insert(0, "05_trajectory_eval")
from corpus import load_staged
from prompts import PROMPT_VERSION, prompt_hash, render_state_prompt
from rules_baseline import evaluate_records
from schema import STATE_VERDICT_SCHEMA, VIOLATION_STATES

print(f"module loaded, judge prompt {PROMPT_VERSION} ({prompt_hash()[:12]})")

In [ ]:
# 20 demo records spread across the step-count distribution, deterministic,
# including at least 3 TRUNCATED records so the mechanic that separates
# Part 2 from Part 1 (head-and-tail windowing, the reduced middle) is
# visible in the demo. Truncation triggers on STEP COUNT, not tokens; the
# smallest truncated records still fit the demo token budget. Staged
# candidates carry PROPOSED labels and adjudicated=false; fine for a demo,
# never fine for a published table (see adjudicate.py).
def total_steps(record):
    return len(record.steps) + len(record.overflow_steps)


records = sorted(load_staged("sample_data/staging/candidates_tier2_batch1.jsonl"), key=total_steps)
untruncated = [r for r in records if r.truncation is None]
truncated = [r for r in records if r.truncation is not None]

mid = len(untruncated) // 2
picks = untruncated[:8] + untruncated[mid : mid + 5] + untruncated[-4:] + truncated[:3]
seen = set()
records = [r for r in picks if not (r.id in seen or seen.add(r.id))][:20]
records.sort(key=total_steps)

counts = [total_steps(r) for r in records]
print(f"{len(records)} records, step counts {counts}")
print(f"{sum(1 for r in records if r.truncation)} truncated records included")

prompts, index = [], []
for record in records:
    for state in VIOLATION_STATES:
        index.append((record.id, state))
        prompts.append(render_state_prompt(record, state))
print(f"-> {len(prompts)} judge prompts (six per record)")

# What a truncated record looks like to the judge: FULL lines carry args
# and output digests, REDUCED lines are the windowed-out middle.
example = next(r for r in records if r.truncation is not None)
lines = render_state_prompt(example, "unsafe_continuation").splitlines()
print("\n--- sample from a truncated record's prompt ---")
print(next(line for line in lines if " FULL " in line))
print(next(line for line in lines if " REDUCED " in line))
print(next(line for line in lines if line.startswith("Truncation:")))

### What the `REDUCED` lines are

Long trajectories keep full fidelity at the **head and tail** (where gates and
claims concentrate) and carry the middle as `REDUCED` steps: actor, action,
target, approval state, stop signals, status, and a shortened claim survive;
per-step args and output digests are dropped for length. The `Truncation:`
line tells the judge exactly what was dropped. In the measured experiment,
misses in the reduced region are scored as a separate population (truncation
artifacts, not judge failures); the head-and-tail design and what it costs
each violation state is one of Part 2's findings.


In [ ]:
# One batched generate() call, structured outputs constrained to the
# per-state verdict schema. max_model_len is 8192 because the truncated
# demo records render long prompts; on free-tier TPU expect several minutes
# of compilation logs before tokens flow. That wait is the compilation
# story from Part 1, in miniature.
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

llm = LLM(model="google/gemma-3-1b-it", max_model_len=8192, dtype="bfloat16")
sampling = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=256,
    structured_outputs=StructuredOutputsParams(json=STATE_VERDICT_SCHEMA),
)
outputs = llm.generate(prompts, sampling)
raw = [output.outputs[0].text for output in outputs]
print(f"{len(raw)} verdicts returned")

In [ ]:
# Put the judge's verdicts next to the deterministic rules baseline: same
# records, same verdict shape, two very different evaluators. No ground
# truth appears here (these records are unadjudicated), so there is
# nothing to compute accuracy against; whether either evaluator is RIGHT
# is exactly what the article's measured tables answer.
import json
from collections import Counter

judge_states = Counter()
judge_examples = []
parse_errors = 0
for (record_id, state), text in zip(index, raw):
    try:
        verdict = json.loads(text)
    except json.JSONDecodeError:
        parse_errors += 1
        continue
    if verdict.get("detected"):
        judge_states[state] += 1
        if len(judge_examples) < 4:
            judge_examples.append((record_id, state, verdict.get("failed_step_index"), verdict.get("evidence", "")))

rules_verdicts = evaluate_records(records)
rules_states = Counter()
rules_examples = []
for record_id, verdict in rules_verdicts.items():
    for violation in verdict["violations"]:
        rules_states[violation["state"]] += 1
        if len(rules_examples) < 3:
            rules_examples.append(
                (record_id, violation["state"], violation["failed_step_index"], violation["evidence"])
            )

print(f"{'state':<26} {'judge':>6} {'rules':>6}")
for state in VIOLATION_STATES:
    print(f"{state:<26} {judge_states[state]:>6} {rules_states[state]:>6}")
print(f"parse errors: {parse_errors} of {len(raw)}")

print("\nJudge detections with evidence (first few):")
for record_id, state, step, evidence in judge_examples:
    print(f"  {record_id} {state}@{step}: {evidence[:110]}")
print("\nRules detections with evidence (first few):")
for record_id, state, step, evidence in rules_examples:
    print(f"  {record_id} {state}@{step}: {evidence[:110]}")

print(
    "\nNote the rules column: zero specification_gaming by design (it abstains),"
    "\nand partial coverage elsewhere. Detection counts on 20 unadjudicated demo"
    "\nrecords; whether either evaluator is right is answered only by the"
    "\narticle's measured tables."
)

## What the full pipeline adds

You just ran the whole Part 2 pattern: bounded canonical trajectories, six independent
judge calls per record under structured outputs, and a deterministic rules baseline in
the same verdict format. The measured experiment differs in scale and rigor, never in
shape:

- `google/gemma-4-E4B-it` on a GCE `v5e-4` via Docker, with XLA length bands selected
  from the corpus token distribution (`05_trajectory_eval/bands.py`)
- A human-adjudicated corpus (`adjudicate.py`; nothing enters it unadjudicated)
- Scoring with tier segmentation, Wilson intervals, and a policy-visible versus
  policy-withheld instrumentation split (`score.py`)

Start at `05_trajectory_eval/README.md` in this repo to run the measured path yourself.
Results in the published article come only from that path.
